# Laboratório — Formatos e Compressão de Imagens com Python

**Duração:** 50 minutos  
**Disciplina:** Tópicos Especiais II — TADS  
**Professor:** Vinícius da Rocha Motta

## Situação-problema

Uma equipe está preparando fotografias, capturas de tela, logotipos e ícones para um portal institucional. O portal deve carregar rapidamente, preservar qualidade visual e manter transparência quando necessário.

Neste laboratório, você produzirá evidências para escolher o formato correto em cada cenário.

> Antes de começar, use **Arquivo → Salvar uma cópia no Drive**.


## Identificação

**Nome(s):** ESCREVA AQUI  
**Data:** ESCREVA AQUI

## Resultados de aprendizagem

Ao concluir, você deverá conseguir comparar JPEG, PNG, WebP e TIFF; explicar compressão com e sem perda; interpretar MSE e PSNR; trabalhar com canal alpha; e explicar a escalabilidade do SVG.


## Cronograma

| Etapa | Tempo |
|---|---:|
| Configuração e geração das imagens | 5 min |
| Inspeção de dimensões e canais | 5 min |
| Comparação de formatos | 15 min |
| JPEG, recortes e métricas | 10 min |
| Transparência | 7 min |
| SVG | 5 min |
| Síntese | 3 min |


# 1. Configuração do ambiente — 5 minutos

A célula abaixo baixa do repositório as funções de apoio e o gerador das imagens. Assim, todos executam o mesmo experimento.


In [ ]:
from pathlib import Path
import sys
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/vmotta/lab-imagens-formatos-python/main"
ARQUIVOS = {
    "image_utils.py": f"{BASE_URL}/src/image_utils.py",
    "gerar_imagens_exemplo.py": f"{BASE_URL}/src/gerar_imagens_exemplo.py",
}

for destino, url in ARQUIVOS.items():
    urllib.request.urlretrieve(url, destino)

sys.path.insert(0, str(Path.cwd()))

from PIL import Image
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import shutil

from image_utils import (
    center_crop,
    ensure_rgb,
    image_metadata,
    mse,
    psnr,
    save_and_measure,
)
from gerar_imagens_exemplo import main as gerar_imagens

PASTA_DADOS = Path("dados")
PASTA_RESULTADOS = Path("resultados")
PASTA_RESULTADOS.mkdir(exist_ok=True)

gerar_imagens(PASTA_DADOS)

print("Ambiente preparado.")


As imagens são geradas por código para tornar o experimento reproduzível:

- uma fotografia sintética;
- uma captura de tela;
- um logotipo com transparência;
- um desenho vetorial SVG.


In [ ]:
foto = Image.open(PASTA_DADOS / "foto_exemplo.png")
tela = Image.open(PASTA_DADOS / "tela_exemplo.png")
logo = Image.open(PASTA_DADOS / "logo_transparente.png")

print("Fotografia sintética")
display(foto.resize((640, 360)))

print("Captura de tela")
display(tela.resize((600, 390)))

print("Logotipo transparente")
display(logo.resize((300, 300)))


# 2. Inspeção da imagem — 5 minutos

Uma imagem rasterizada é uma grade de pixels. Dimensão, total de pixels, modo de cor e tamanho do arquivo representam propriedades diferentes.


In [ ]:
metadados = pd.DataFrame([
    {"imagem": "fotografia", **image_metadata(foto)},
    {"imagem": "captura de tela", **image_metadata(tela)},
    {"imagem": "logotipo", **image_metadata(logo)},
])

display(metadados)


### Questão 1 — RESPOSTA

Compare os modos de cor das três imagens. Qual possui canal alpha? Explique o que esse canal representa.

**Resposta:** ESCREVA AQUI


# 3. Comparação entre formatos — 15 minutos

Vamos salvar a mesma fotografia em vários formatos. Para uma comparação correta, a imagem permanece com as mesmas dimensões.


In [ ]:
foto_rgb = ensure_rgb(foto)

experimentos = [
    ("foto_jpeg_q95.jpg", "JPEG", {"quality": 95, "optimize": True}),
    ("foto_jpeg_q60.jpg", "JPEG", {"quality": 60, "optimize": True}),
    ("foto_jpeg_q20.jpg", "JPEG", {"quality": 20, "optimize": True}),
    ("foto_png.png", "PNG", {"optimize": True}),
    ("foto_webp_q80.webp", "WEBP", {"quality": 80}),
    ("foto_webp_lossless.webp", "WEBP", {"lossless": True}),
    ("foto_tiff_lzw.tiff", "TIFF", {"compression": "tiff_lzw"}),
]

resultados = []
for nome, formato, opcoes in experimentos:
    registro = save_and_measure(
        foto_rgb,
        PASTA_RESULTADOS / nome,
        formato,
        **opcoes,
    )
    resultados.append(registro)

tabela_tamanhos = (
    pd.DataFrame(resultados)
    .sort_values("tamanho_kb")
    .reset_index(drop=True)
)

display(tabela_tamanhos)


In [ ]:
ax = tabela_tamanhos.plot.barh(
    x="arquivo",
    y="tamanho_kb",
    legend=False,
    figsize=(9, 5),
)
ax.set_xlabel("Tamanho do arquivo (KB)")
ax.set_ylabel("")
ax.set_title("Mesma imagem e mesmas dimensões em diferentes formatos")
plt.tight_layout()
plt.show()


### Questão 2 — RESPOSTA

1. Qual arquivo ficou menor?
2. Qual ficou maior?
3. PNG ficou necessariamente menor que JPEG?
4. Qual opção parece mais adequada para essa fotografia na web?
5. Por que não é correto dizer que um formato sempre produz o menor arquivo?

**Resposta:** ESCREVA AQUI


## Fotografia versus captura de tela

O resultado da compressão também depende do conteúdo. Vamos comparar JPEG e PNG usando uma captura de tela com textos e bordas.


In [ ]:
tela_rgb = ensure_rgb(tela)

comparacao_conteudo = []
for origem, imagem in [("fotografia", foto_rgb), ("captura de tela", tela_rgb)]:
    for nome, formato, opcoes in [
        (f"{origem}_q80.jpg", "JPEG", {"quality": 80, "optimize": True}),
        (f"{origem}.png", "PNG", {"optimize": True}),
    ]:
        item = save_and_measure(
            imagem,
            PASTA_RESULTADOS / nome,
            formato,
            **opcoes,
        )
        item["conteudo"] = origem
        comparacao_conteudo.append(item)

display(pd.DataFrame(comparacao_conteudo))


### Questão 3 — RESPOSTA

O comportamento de JPEG e PNG foi igual para a fotografia e para a captura de tela? Relacione sua resposta ao tipo de conteúdo presente em cada imagem.

**Resposta:** ESCREVA AQUI


# 4. Compressão JPEG e métricas — 10 minutos

JPEG usa compressão com perda. Qualidades menores tendem a descartar mais informação.


In [ ]:
nomes_jpeg = [
    "foto_jpeg_q95.jpg",
    "foto_jpeg_q60.jpg",
    "foto_jpeg_q20.jpg",
]

for nome in nomes_jpeg:
    imagem = Image.open(PASTA_RESULTADOS / nome).convert("RGB")
    recorte = center_crop(imagem, 0.45)
    recorte = recorte.resize((recorte.width * 2, recorte.height * 2))
    print(nome)
    display(recorte)


Observe especialmente bordas, letras, áreas de cor suave e regiões com textura. Procure borrões, marcas em bloco e perda de nitidez.


In [ ]:
metricas = []

for nome in nomes_jpeg:
    comprimida = Image.open(PASTA_RESULTADOS / nome).convert("RGB")
    metricas.append({
        "arquivo": nome,
        "tamanho_kb": round((PASTA_RESULTADOS / nome).stat().st_size / 1024, 2),
        "MSE": round(mse(foto_rgb, comprimida), 2),
        "PSNR_dB": round(psnr(foto_rgb, comprimida), 2),
    })

tabela_metricas = pd.DataFrame(metricas)
display(tabela_metricas)


### Como interpretar

- **MSE menor:** menos diferença média entre os pixels.
- **PSNR maior:** maior semelhança numérica com a imagem original.
- As métricas auxiliam a análise, mas não substituem a observação visual e a finalidade da imagem.


### Questão 4 — RESPOSTA

Explique a relação observada entre qualidade JPEG, tamanho do arquivo, MSE e PSNR. Em qual nível os defeitos se tornaram evidentes para você?

**Resposta:** ESCREVA AQUI


# 5. Transparência e canal alpha — 7 minutos

PNG e WebP podem armazenar transparência. JPEG não possui canal alpha.


In [ ]:
alpha = logo.getchannel("A")

plt.figure(figsize=(6, 6))
plt.imshow(alpha, cmap="gray")
plt.title("Canal alpha — branco: opaco; preto: transparente")
plt.axis("off")
plt.show()


In [ ]:
transparencia_resultados = []

transparencia_resultados.append(
    save_and_measure(
        logo,
        PASTA_RESULTADOS / "logo_alpha.png",
        "PNG",
        optimize=True,
    )
)

transparencia_resultados.append(
    save_and_measure(
        logo,
        PASTA_RESULTADOS / "logo_alpha.webp",
        "WEBP",
        lossless=True,
    )
)

logo_com_fundo = ensure_rgb(logo, background=(255, 255, 255))
transparencia_resultados.append(
    save_and_measure(
        logo_com_fundo,
        PASTA_RESULTADOS / "logo_sem_alpha.jpg",
        "JPEG",
        quality=90,
        optimize=True,
    )
)

display(pd.DataFrame(transparencia_resultados))

print("PNG")
display(Image.open(PASTA_RESULTADOS / "logo_alpha.png").resize((280, 280)))

print("WebP")
display(Image.open(PASTA_RESULTADOS / "logo_alpha.webp").resize((280, 280)))

print("JPEG composto sobre fundo branco")
display(Image.open(PASTA_RESULTADOS / "logo_sem_alpha.jpg").resize((280, 280)))


### Questão 5 — RESPOSTA

O que aconteceu com a transparência na versão JPEG? Por que PNG ou WebP são mais adequados para logotipos com fundo transparente?

**Resposta:** ESCREVA AQUI


# 6. SVG e escalabilidade — 5 minutos

SVG descreve formas geométricas, textos, linhas e curvas. Ele não armazena a imagem como uma grade fixa de pixels.


In [ ]:
svg_texto = (PASTA_DADOS / "icone_escalavel.svg").read_text(encoding="utf-8")

display(HTML(f'''
<h3>SVG com largura de 240 pixels</h3>
<div style="width:240px">{svg_texto}</div>

<h3>O mesmo SVG com largura de 900 pixels</h3>
<div style="width:900px; max-width:100%">{svg_texto}</div>
'''))


### Questão 6 — RESPOSTA

Por que o SVG permanece nítido quando é ampliado? Em qual situação você escolheria SVG e em qual situação escolheria um formato rasterizado?

**Resposta:** ESCREVA AQUI


# 7. Síntese e decisão técnica — 3 minutos

Preencha a matriz de decisão usando as evidências obtidas.


| Situação | Formato escolhido | Justificativa baseada no experimento |
|---|---|---|
| Fotografia para página web | ESCREVA | ESCREVA |
| Logotipo com fundo transparente | ESCREVA | ESCREVA |
| Captura de tela com textos | ESCREVA | ESCREVA |
| Fotografia para edição profissional | ESCREVA | ESCREVA |
| Ícone exibido em vários tamanhos | ESCREVA | ESCREVA |
| Imagem científica com preservação de detalhes | ESCREVA | ESCREVA |


## Conclusão — RESPOSTA

Em aproximadamente cinco linhas, responda:

> Existe um único formato de imagem que seja o melhor para todas as situações?

Use pelo menos duas evidências produzidas no laboratório.

**Resposta:** ESCREVA AQUI


# 8. Exportação dos resultados

A célula abaixo cria um ZIP com as imagens produzidas. O arquivo serve para conferência; a entrega principal continua sendo o notebook preenchido.


In [ ]:
arquivo_zip = shutil.make_archive(
    "resultados_imagens",
    "zip",
    PASTA_RESULTADOS,
)

print("Arquivo criado:", arquivo_zip)

try:
    from google.colab import files
    files.download(arquivo_zip)
except ImportError:
    print("Execução local: o ZIP está na pasta atual.")


## Checklist de entrega

- [ ] Preenchi meu nome.
- [ ] Executei todas as células.
- [ ] Respondi às seis questões.
- [ ] Completei a matriz de decisão.
- [ ] Escrevi a conclusão.
- [ ] Salvei uma cópia do notebook.
